# 02 · Training EfficientNet-B0 (leakage-aware and naive splits)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rishabh-bgp/brain-tumor-mri-classification/blob/main/notebooks/02_training_efficientnet_b0.ipynb)

Fine-tunes an ImageNet-pretrained EfficientNet-B0 (timm) on 224×224 greyscale slices. The *leakage-aware* run uses the group-based split from notebook 01; the *naive* run uses the original Training folder with a random image-level validation split, for comparison. Set `FORCE_TRAIN = True` to retrain (about 30 min per run on an 8-core CPU, a few minutes on a Colab GPU). When checkpoints already exist the notebook shows the logged history of the run that produced the committed results.

*Rishabh Aryan · M.Tech (AI & Data Science), IIIT Bhagalpur. Research and education code, not a medical device.*

In [ ]:
# --- Setup: works locally (repo checkout) and on Google Colab ---
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("brain-tumor-mri-classification"):
        subprocess.run(["git", "clone", "-q", "https://github.com/Rishabh-bgp/brain-tumor-mri-classification.git"], check=True)
    os.chdir("brain-tumor-mri-classification")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "kagglehub", "imagehash", "captum",
                    "onnx", "onnxruntime", "scikit-image"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("working directory:", os.getcwd())

In [ ]:
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (Colab: Runtime > Change runtime type > GPU for faster training)")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

In [ ]:
FORCE_TRAIN = False
from pathlib import Path
for run in ("clean", "naive"):
    ckpt = Path(f"results/runs/{run}/best.pt")
    if FORCE_TRAIN or not ckpt.exists():
        subprocess.run([sys.executable, "-m", "btmri.train", "--run", run, "--epochs", "12"], cwd="src", check=True)
    else:
        print(f"[{run}] using existing checkpoint {ckpt} (set FORCE_TRAIN = True to retrain)")

In [ ]:
import pandas as pd
for run in ("clean", "naive"):
    h = pd.read_csv(f"results/runs/{run}/history.csv")
    print(f"=== {run} run: best epoch by validation macro-F1 = {int(h.loc[h.val_macro_f1.idxmax(), 'epoch'])}")
    display(h.round(4))

In [ ]:
from IPython.display import Image, display
for run in ("clean", "naive"):
    display(Image(f"results/figures/training_curves_{run}.png", width=760))

**Training recipe** (see `src/btmri/train.py`): AdamW (lr 5e-4, weight decay 1e-4), one-cycle schedule, batch 32, 12 epochs, inverse-frequency class weights, augmentation = random resized crop (scale 0.8–1), horizontal flip, ±10° rotation / 5% translation, brightness/contrast jitter 0.2; seed 42; the checkpoint with the best validation macro-F1 is kept.